# Dusha Fine-Tune — WavLM Base Full Fine-Tune (Kaggle)

Trains `xbgoose/wavlm-base-speech-emotion-recognition-russian-dusha-finetuned`  
on crowd-aggregated Dusha labels (Dawid-Skene or Majority Vote).

**Input в Kaggle:**
- `dusha-datasetcrowd` — аудиофайлы (wavs/)
- твой датасет с агрегированным TSV (aggregated_*.tsv)

**Выбор разметки:** поменяй `AGGREGATED_TSV` ниже.

## 1. GPU check

In [ ]:
import subprocess, sys, os
import torch

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("CUDA not available")

## 2. Install dependencies

In [ ]:
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'torchaudio', 'transformers>=4.40', 'datasets>=2.18', 'peft>=0.10',
    'scikit-learn', 'matplotlib', 'seaborn', 'soundfile', 'pyyaml', 'tqdm',
], check=True)
print('Done.')

## 3. Clone repo

In [ ]:
REPO_URL = 'https://github.com/aibryanov/speech_emo_finetune.git'
REPO_DIR = 'speech_emo_finetune'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
else:
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)

os.chdir(REPO_DIR)
if os.getcwd() not in sys.path:
    sys.path.insert(0, os.getcwd())
print('Working directory:', os.getcwd())

## 4. Config

Укажи имя датасета с агрегированными TSV — обучится по очереди для каждой агрегации.

In [ ]:
import warnings, logging, pathlib
warnings.filterwarnings('ignore')
logging.getLogger('transformers').setLevel(logging.ERROR)

from src.config import load_config

# ── укажи имя своего датасета с агрегированными TSV ──────────────────────────
AGGREGATED_DATASET = '<твой-датасет>'   # имя датасета в Kaggle input
# ─────────────────────────────────────────────────────────────────────────────

AGG_ROOT = pathlib.Path(f'/kaggle/input/{AGGREGATED_DATASET}')

# все 5 агрегаций
AGGREGATIONS = {
    'majority':  AGG_ROOT / 'aggregated_majority.tsv',
    'ds_0.85':   AGG_ROOT / 'aggregated_ds_0.85.tsv',
    'ds_0.9':    AGG_ROOT / 'aggregated_ds_0.9.tsv',
    'ds_0.95':   AGG_ROOT / 'aggregated_ds_0.95.tsv',
    'ds_0.98':   AGG_ROOT / 'aggregated_ds_0.98.tsv',
}

# автодетект пути к аудио
_candidates = [
    '/kaggle/input/dusha-datasetcrowd',
    '/kaggle/input/dusha-datasetcrowd/dusha-datasetcrowd',
]
AUDIO_DIR = next(
    (p for p in _candidates if (pathlib.Path(p) / 'wavs').exists()),
    _candidates[0],
)
print(f'audio_dir : {AUDIO_DIR}')
print(f'wavs/     : {(pathlib.Path(AUDIO_DIR) / "wavs").exists()}')
print()

# GPU info
import torch
n_gpus = torch.cuda.device_count()
print(f'GPUs available: {n_gpus}')
for i in range(n_gpus):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)}  '
          f'{torch.cuda.get_device_properties(i).total_memory/1e9:.1f} GB')

# проверяем какие TSV существуют
import pandas as pd
print()
for tag, path in AGGREGATIONS.items():
    exists = path.exists()
    n = len(pd.read_csv(path, sep='\t')) if exists else 0
    print(f'  {tag:12s}  {"✓" if exists else "✗"}  {n:6,} rows  {str(path)}')

## 5. Train all 5 models

In [ ]:
import gc, random
import numpy as np
import torch.nn as nn
from transformers import AutoFeatureExtractor

from src.dataset import get_dusha_dataloaders
from src.models import build_model
from src.trainer import Trainer

def set_seed(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_dp = torch.cuda.device_count() >= 2
print(f'Device: {device}  |  DataParallel: {use_dp} ({torch.cuda.device_count()} GPUs)')

all_results = {}

for tag, tsv_path in AGGREGATIONS.items():
    if not tsv_path.exists():
        print(f'\n[{tag}] TSV not found — skip')
        continue

    print(f'\n{"="*60}')
    print(f'  Training: {tag}  ({tsv_path.name})')
    print(f'{"="*60}')

    # cleanup
    for _var in ['train_loader', 'dev_loader', 'trainer', 'model']:
        if _var in dir():
            del _var
    gc.collect()
    torch.cuda.empty_cache()

    config = load_config('configs/wavlm_base_dusha_full.yaml')
    config.aggregated_tsv = str(tsv_path)
    config.audio_dir      = AUDIO_DIR
    config.num_workers    = 0
    config.run_name       = f'wavlm_base_dusha_{tag}'
    config.output_dir     = f'outputs/wavlm_base_dusha_{tag}'
    set_seed(config.seed)

    processor = AutoFeatureExtractor.from_pretrained(
        config.processor_name or config.model_name
    )
    train_loader, dev_loader, _ = get_dusha_dataloaders(config, processor)
    print(f'Train: {len(train_loader.dataset):,} | Dev: {len(dev_loader.dataset):,}', flush=True)

    model = build_model(config)
    if use_dp:
        model = nn.DataParallel(model)
        print(f'DataParallel: {torch.cuda.device_count()} GPUs', flush=True)

    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f'Params: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)', flush=True)

    trainer = Trainer(model, config, train_loader, dev_loader, dev_loader, device)
    metrics = trainer.fit()
    all_results[tag] = metrics

    # показываем сохранённые файлы
    out = pathlib.Path(config.output_dir)
    print(f'\nSaved to {out}:')
    for f in sorted(out.iterdir()):
        print(f'  {f.name:45s}  {f.stat().st_size/1e6:6.1f} MB')

print('\n\nAll done!')

## 6. Compare all 5 models

In [ ]:
import json, pathlib
import matplotlib.pyplot as plt
import pandas as pd

rows = []
for tag in AGGREGATIONS:
    mpath = pathlib.Path(f'outputs/wavlm_base_dusha_{tag}/metrics.jsonl')
    if not mpath.exists():
        continue
    recs = [json.loads(l) for l in open(mpath)]
    test_rec = [r for r in recs if r.get('split') == 'test']
    if not test_rec:
        continue
    t = test_rec[-1]
    rows.append({
        'aggregation': tag,
        'accuracy': t['accuracy'],
        'weighted_accuracy': t['weighted_accuracy'],
        'f1_macro': t['f1_macro'],
        'f1_weighted': t['f1_weighted'],
    })

df = pd.DataFrame(rows).set_index('aggregation')
print(df.to_string(float_format='{:.4f}'.format))

df[['accuracy', 'f1_macro', 'f1_weighted']].plot(kind='bar', figsize=(10, 5))
plt.title('WavLM Base Dusha — по агрегациям')
plt.ylabel('Score')
plt.xticks(rotation=20, ha='right')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()